<a href="https://colab.research.google.com/github/todd-jang/AIFFEL_quest_rs/blob/main/GoingDeeper/06_Transformer/project_3_2_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === Colab 호환 셋업 (자동 추가) ===
import torch as _t
_t._orig_load = getattr(_t, '_orig_load', _t.load)
def _compat_load(*a, **k):
    k.setdefault('weights_only', False)
    return _t._orig_load(*a, **k)
_t.load = _compat_load
try:
    import matplotlib; matplotlib.rcParams['axes.unicode_minus'] = False
except Exception: pass
print('[colab-compat] torch.load weights_only=False 패치 적용')


[colab-compat] torch.load weights_only=False 패치 적용


In [2]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 63.6 MB/s eta 0:00:00


# **17. 번역가는 대화에도 능하다**

## **17-1. 들어가며**

좋은 번역을 만드는 데에는 무슨 능력이 필요할까요? 가장 먼저 떠오르는 것은 역시 언어 능력이죠! 적어도 번역하고자 하는 언어는 통달해야 좋은 번역을 해낼 수 있을 것 같습니다. 하지만 뛰어난 언어 실력만으로 가능할까요? \
\
`"Lost In Translation"`은 동명의 영화로 유명해진 말인데요, 번역이 언어적 의미 너머의 맥락과 함의 또한 유실 없이 전달해야 함을 시사합니다. 동시에 문화적 차이가 존재하는 한 절대 사라질 수 없는 말이기도 하죠. 번역가들은 이 **Lost In Translation**을 최소화하기 위해 자신과의 싸움을 하고, 그렇게 탄생한 멋진 결과물은 한글 패치 잘 되었다는 극찬을 받게 됩니다. ^_^ \
\
말하고 싶은 것은, 번역가들의 번역이 단순히 언어를 변환하는 과정에 그치는 것이 아니라 원문을 이해하고 그 이해를 바탕으로 새로운 글을 작문하여 탄생한다는 겁니다. 그렇기에 번역에 능숙한 이들은 대체로 언변도 좋고, 대화에도 능합니다. 언어적 이해 능력이 뛰어나니까요! 번역가의 멋진 면모를 볼 수 있는 재미난 영상을 하나 첨부해드리니, 시간 날 때 가볍게 살펴보세요 😃

[![image.png](https://img.youtube.com/vi/8zfYINYNS38/0.jpg)](https://youtu.be/8zfYINYNS38)

인공지능도 마찬가지입니다. 번역을 잘 해낼 수 있는 모델은 곧 언어를 잘 이해할 수 있는 모델이기도 해요. 그래서 번역을 잘하는 트랜스포머가 자언어 이해(Natural Language Understanding) 모델의 근간이 되는 거죠! **질문과 답변을 주고받는 것** 또한 제법 높은 수준의 자연어 이해를 요구하는데, 이것도 잘 해낼 수 있을지 이번 코스에서 함께 확인해 보도록 해요. **번역 모델을 활용한 챗봇 만들기!** 얼른 시작해 볼까요?

**[아이스브레이킹] 번역용 데이터와 챗봇 데이터의 차이점은 무엇이 있을까요?**<br>
A. [ 답변을 적어볼까요! ]<br><br>

<details><summary>💡예시답안 확인하기💡</summary>

번역 데이터는 피번역어(소스 문장)와 번역어(타깃 문장)으로 구성되어있다면, 챗봇 데이터는 질문(소스 문장)과 답변(타깃 문장)으로 구성되어있겠죠?</details>

### **학습 내용**
---

* 2. 번역 데이터 준비
  * 번역을 위해 영어-스페인어 데이터셋을 사용해보아요.
* 3. 번역 모델 만들기
  * 번역엔 뭐다? Transformer다!
* 4. 번역 성능 측정하기 (1) BLEU Score
  * 몇 점이면 훌륭한 번역기라고 할 수 있을까요?
* 5. 번역 성능 측정하기 (2) Beam Search Decoder
  * Beam search + BLEU = ?
* 6. 데이터 부풀리기
  * 내 모델을 강하고 똑똑하게 만들어 보아요.

### **학습 목표**
---

* 번역 및 챗봇 성능을 측정하기 위한 지표를 이해하고, 용도에 맞게 만들 수 있다.
* NLP task에 맞는 data augmentation의 방법들을 알고, 활용할 수 있다.

### **준비물**
---

터미널을 열고 프로젝트를 위한 디렉토리를 생성해 주세요.

In [3]:
!mkdir -p ./aiffel/transformer_chatbot

아직 KoNLPy가 설치되어 있지 않으시다면, 우분투 환경에서는 아래 소스를 실행하여 설치해 주시고, 다른 OS는 첨부한 공식 문서를 참고하여 설치하시길 바랍니다.

###### Ubuntu

```shell
$ sudo apt-get install g++ openjdk-8-jdk
$ sudo apt-get install curl

$ bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

$ pip install konlpy
```

###### Windows, Mac

* [설치하기 - KoNLPy 0.5.2 documentation](https://konlpy.org/ko/latest/install/)

In [4]:
import os

!apt-get update
!apt-get install g++ openjdk-8-jdk python3-dev python3-pip curl
!python3 -m pip install --upgrade pip

# Run mecab.sh to install MeCab system libraries and dictionary
# !bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

# Install MeCab Python bindings and konlpy together for better compatibility
!pip install mecab-python3 konlpy

print('MeCab and Konlpy installation process updated.')

# Verify MeCab installation
try:
    from konlpy.tag import Mecab
    mecab = Mecab()
    print('MeCab is successfully initialized by Konlpy.')
except Exception as e:
    print(f'Error initializing MeCab: {e}')
    print('Please check the MeCab installation steps carefully.')

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,832 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main 

## **17-2. 번역 데이터 준비**

**Please restart the Colab runtime (Runtime > Restart runtime) after the previous cell has finished executing to ensure all new packages are loaded correctly.**

먼저 번역 모델이 있어야 챗봇을 만들 수 있겠죠? 이번 실습에선 접근성이 좋은 **영어-스페인어 데이터**를 사용하도록 하겠습니다.

### **라이브러리와 데이터 준비하기**
---

필요한 라이브러리를 `import` 해주세요.

In [5]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.11.0+cu128


영어-스페인어 데이터를 다운로드해 주세요.

In [6]:
import urllib.request
import zipfile

zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

print("슝=3")

슝=3


다운로드한 데이터가 어떤 지 한 번 열어봅시다. 중복된 데이터가 있을 수 있으니 `list`와 `set`을 사용해 처리해 줍니다.

In [7]:
extracted_folder = "./spa-eng"
file_path = os.path.join(extracted_folder, "spa.txt")

with open(file_path, "r") as f:
    spa_eng_sentences = f.read().splitlines()

spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(spa_eng_sentences)
print("Example:", total_sentence_count)

for sen in spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Example: 118964
>> Answer in English.	Responde en inglés.
>> Are you studying?	¿Estás estudiando?
>> He's smarter than they are.	Él es más inteligente que ellos.
>> He is an expert at fishing.	Él es un experto pescando.
>> Did you miss me?	¿Me extrañaste?


한 줄에 영어와 스페인어가 둘 다 들어 있네요. \
\
가벼운 전처리를 해줍시다. 먼저 전처리 함수를 만들어 볼까요?

In [8]:
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

모든 데이터에 대해서 같은 전처리를 해줄게요.

In [9]:
spa_eng_sentences = list(map(preprocess_sentence, spa_eng_sentences))

print('슝=3')

슝=3


이제 테스트에 사용할 데이터를 따로 떼어냅니다. 전체 데이터의 0.5% 정도를 테스트용으로 사용할게요.

In [10]:
test_sentence_count = total_sentence_count // 200
print("Test Size: ", test_sentence_count)
print("\n")

train_spa_eng_sentences = spa_eng_sentences[:-test_sentence_count]
test_spa_eng_sentences = spa_eng_sentences[-test_sentence_count:]
print("Train Example:", len(train_spa_eng_sentences))
for sen in train_spa_eng_sentences[0:100][::20]:
    print(">>", sen)
print("\n")
print("Test Example:", len(test_spa_eng_sentences))
for sen in test_spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Test Size:  594


Train Example: 118370
>> answer in english.	responde en inglés.
>> are you studying?	¿estás estudiando?
>> he's smarter than they are.	él es más inteligente que ellos.
>> he is an expert at fishing.	él es un experto pescando.
>> did you miss me?	¿me extrañaste?


Test Example: 594
>> when i grow up, i want to be a king.	quiero ser rey cuando sea grande.
>> don't say that.	¡no digas eso!
>> has tom become crazy?	¿tom se volvió loco?
>> remove your hat.	quítate el sombrero.
>> i want a divorce.	quiero el divorcio.


한 줄에 포함되어 있는 영어와 스페인어를 분리해 줍니다. 영어 문장과 스페인어 문장이 tab으로 연결되어 있으니 `split('\t')`을 사용하면 나눌 수 있겠네요. tab 이전이 영어, 이후가 스페인어 문장입니다. \
\
먼저 함수를 만들어 줍니다.

In [11]:
def split_spa_eng_sentences(spa_eng_sentences):
    spa_sentences = []
    eng_sentences = []
    for spa_eng_sentence in tqdm(spa_eng_sentences):
        eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        spa_sentences.append(spa_sentence)
        eng_sentences.append(eng_sentence)
    return eng_sentences, spa_sentences

print('슝=3')

슝=3


학습 데이터와 테스트 데이터를 모두 나눠 줍니다.

In [12]:
train_eng_sentences, train_spa_sentences = split_spa_eng_sentences(train_spa_eng_sentences)
print(len(train_eng_sentences))
print(train_eng_sentences[0])
print('\n')
print(len(train_spa_sentences))
print(train_spa_sentences[0])

  0%|          | 0/118370 [00:00<?, ?it/s]

118370
answer in english.


118370
responde en inglés.


In [13]:
test_eng_sentences, test_spa_sentences = split_spa_eng_sentences(test_spa_eng_sentences)
print(len(test_eng_sentences))
print(test_eng_sentences[0])
print('\n')
print(len(test_spa_sentences))
print(test_spa_sentences[0])

  0%|          | 0/594 [00:00<?, ?it/s]

594
when i grow up, i want to be a king.


594
quiero ser rey cuando sea grande.


### **토큰화**
---

이제 문장 데이터를 토큰화를 해야 할 차례입니다. **Sentencepiece** 기반의 토크나이저를 생성해 주는 `generate_tokenizer()` 함수를 정의하여 토크나이저를 얻어보도록 하죠!<br><br>

* [google/sentencepiece](https://github.com/google/sentencepiece)

In [14]:
def generate_tokenizer(corpus,
                       vocab_size,
                       lang="spa-eng",
                       pad_id=0,   # pad token의 일련번호
                       bos_id=1,  # 문장의 시작을 의미하는 bos token(<s>)의 일련번호
                       eos_id=2,  # 문장의 끝을 의미하는 eos token(</s>)의 일련번호
                       unk_id=3):   # unk token의 일련번호
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=./%s --model_prefix=%s --vocab_size=%d '\
        % (file, model, vocab_size) + \
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'\
        % (pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer

print("슝=3")

슝=3


이번엔 한-영 번역 때와 다르게, **두 언어가 단어 사전을 공유**하도록 하겠습니다. 영어와 스페인어 **모두 알파벳**으로 이뤄지는 데다가 같은 **인도유럽어족**이기 때문에 기대할 수 있는 효과가 많아요! 후에 챗봇을 만들 때에도 질문과 답변이 모두 한글로 이루어져 있기 때문에 Embedding 층을 공유하는 것이 성능에 도움이 됩니다. \
\
단어 사전 수는 **20,000**으로 설정하겠습니다. 처리하는데 약간 시간이 걸립니다.

In [15]:
VOCAB_SIZE = 20000
tokenizer = generate_tokenizer(train_eng_sentences + train_spa_sentences, VOCAB_SIZE, 'spa-eng')
# tokenizer.set_encode_extra_options("bos:eos")  # 문장 양 끝에 <s> , </s> 추가

위에서 두 언어 사이에 단어 사전을 공유하기로 하였으므로 Encoder와 Decoder의 전용 토크나이저를 만들지 않고, 방금 만들어진 토크나이저를 두 언어 사이에서 공유하게 됩니다. \
\
토크나이저가 준비되었으니 본격적으로 데이터를 토큰화하도록 하겠습니다. 토큰화를 해주는 함수를 만들어 줍니다.

In [16]:
def make_corpus(sentences, tokenizer):
    corpus = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.encode_as_ids(sentence)
        corpus.append(tokens)
    return corpus

print('슝=3')

슝=3


영어와 스페인어를 각각 토큰화 해줍니다. 훈련 데이터만 토큰화를 하고, 같은 토크나이저를 사용한다는 점에 주의하세요.

In [17]:
eng_corpus = make_corpus(train_eng_sentences, tokenizer)
spa_corpus = make_corpus(train_spa_sentences, tokenizer)

  0%|          | 0/118370 [00:00<?, ?it/s]

  0%|          | 0/118370 [00:00<?, ?it/s]

토큰화가 잘 되었는지 확인해 봅시다.

In [18]:
print(train_eng_sentences[0])
print(eng_corpus[0])
print('\n')
print(train_spa_sentences[0])
print(spa_corpus[0])

answer in english.
[546, 29, 343, 4]


responde en inglés.
[6043, 24, 350, 4]


`list` 자료형을 고정 길이로 맞추기 위해 패딩(padding) 작업을 해줍니다. 기존에는 `tf.keras`의 `pad_sequences()`를 사용했지만, 여기서는 PyTorch에 맞춰 직접 구현한 `pad_sequences_custom()`으로 한 번에 데이터셋을 완성하겠습니다! 한 문장의 토큰 길이가 50이 되도록 설정했습니다.

In [19]:
# MAX_LEN = 50
# enc_ndarray = tf.keras.preprocessing.sequence.pad_sequences(eng_corpus, maxlen=MAX_LEN, padding='post')
# dec_ndarray = tf.keras.preprocessing.sequence.pad_sequences(spa_corpus, maxlen=MAX_LEN, padding='post')

# print('슝=3')

In [20]:
MAX_LEN = 50

def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    """
    sequences: list of list (각 문장별 토큰 ID 리스트)
    max_len: 고정할 최대 시퀀스 길이
    pad_value: 패딩에 사용할 값 (일반적으로 0)
    """
    padded_sequences = []

    for seq in sequences:
        # 초과 길이는 자르고
        if len(seq) > max_len:
            seq = seq[:max_len]
        # 부족한 길이는 pad_value로 채우기
        else:
            seq = seq + [pad_value] * (max_len - len(seq))

        padded_sequences.append(seq)

    # 최종적으로 torch.Tensor로 변환 (shape: [batch_size, max_len])
    return torch.tensor(padded_sequences, dtype=torch.long)

enc_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)
dec_ndarray = pad_sequences_custom(spa_corpus, max_len=MAX_LEN, pad_value=0)

print(enc_ndarray.shape)  # 예) [batch_size, 50]
print(dec_ndarray.shape)  # 예) [batch_size, 50]
print("슝=3")

torch.Size([118370, 50])
torch.Size([118370, 50])
슝=3


이제 모델 훈련에 사용될 수 있도록 영어와 스페인어 데이터를 묶어 배치 크기의 텐서로 만들어 줍니다. 데이터 셋이 완성 되었어요!

In [21]:
# BATCH_SIZE = 64
# train_dataset = tf.data.Dataset.from_tensor_slices((enc_ndarray, dec_ndarray)).batch(batch_size=BATCH_SIZE)

# print('슝=3')

In [22]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = TensorDataset(enc_ndarray, dec_ndarray)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print("슝=3")

슝=3


이제 모델을 만들러 가봅시다~!

## **17-3. 번역 모델 만들기**

### **트랜스포머 구현하기**
---

**생성된 데이터를 학습할 수 있는 멋진 트랜스포머(Transformer)를 구현하세요!** \
\
트랜스포머 구조가 잘 기억나지 않으시거나 구현에 도움이 필요하시면 아래 링크를 참고해 주세요.<br><br>

* [위키독스: 트랜스포머](https://wikidocs.net/31379)
* [Trax: Transformer](https://github.com/google/trax/blob/master/trax/models/transformer.py)
* [`nn.Transformer` 와 torchtext로 시퀀스-투-시퀀스(Sequence-to-Sequence) 모델링하기](https://tutorials.pytorch.kr/beginner/transformer_tutorial.html)

<br>단, Encoder와 Decoder 각각의 Embedding과 출력층의 Linear, 총 3개의 레이어가 Weight를 공유할 수 있게 하세요! \
\
하이퍼파라미터는 아래와 동일하게 정의합니다.

```python
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)
```

아래 코드 블록에 모듈별로 하나씩 구현해 봅시다.

###### Positional Encoding

In [23]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
print("슝=3")

슝=3


<details><summary>예시 답안</summary>

```python
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
```
</details>

###### 마스크 생성

In [24]:
# # Mask  생성하기
# def generate_padding_mask(seq):
#     seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
#     return seq[:, tf.newaxis, tf.newaxis, :]

# def generate_lookahead_mask(size):
#     mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
#     return mask

# def generate_masks(src, tgt):
#     enc_mask = generate_padding_mask(src)
#     dec_enc_mask = generate_padding_mask(src)

#     dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])
#     dec_tgt_padding_mask = generate_padding_mask(tgt)
#     dec_mask = tf.maximum(dec_tgt_padding_mask, dec_lookahead_mask)

#     return enc_mask, dec_enc_mask, dec_mask
# print("슝=3")

<details><summary>예시 답안</summary>

```python
def generate_padding_mask(seq):
    # (seq == 0)인 위치를 1로 표시 -> [batch, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()

def generate_lookahead_mask(size):
    # 주대각선 위쪽(미래 토큰)을 1로 채운다
    return torch.triu(torch.ones(size, size), diagonal=1)

def generate_masks(src, tgt):
    enc_mask = generate_padding_mask(src)
    dec_enc_mask = generate_padding_mask(src)

    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])
    dec_tgt_padding_mask = generate_padding_mask(tgt)
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)

    dec_mask = torch.max(dec_tgt_padding_mask.to(device),
                         dec_lookahead_mask.to(device))
    return enc_mask, dec_enc_mask, dec_mask
```
</details>


In [25]:
import torch

def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    enc_mask = generate_padding_mask(src)
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    dec_enc_mask = generate_padding_mask(src)

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

슝=3


###### Multi-head Attention

In [26]:
# # Multi Head Attention 구현
# class MultiHeadAttention(tf.keras.layers.Layer):
#     def __init__(self, d_model, num_heads):
#         super(MultiHeadAttention, self).__init__()
#         self.num_heads = num_heads
#         self.d_model = d_model

#         self.depth = d_model // self.num_heads

#         self.W_q = tf.keras.layers.Dense(d_model)
#         self.W_k = tf.keras.layers.Dense(d_model)
#         self.W_v = tf.keras.layers.Dense(d_model)

#         self.linear = tf.keras.layers.Dense(d_model)

#     def scaled_dot_product_attention(self, Q, K, V, mask):
#         d_k = tf.cast(K.shape[-1], tf.float32)
#         QK = tf.matmul(Q, K, transpose_b=True)

#         scaled_qk = QK / tf.math.sqrt(d_k)

#         if mask is not None: scaled_qk += (mask * -1e9)

#         attentions = tf.nn.softmax(scaled_qk, axis=-1)
#         out = tf.matmul(attentions, V)

#         return out, attentions


#     def split_heads(self, x):
#         bsz = x.shape[0]
#         split_x = tf.reshape(x, (bsz, -1, self.num_heads, self.depth))
#         split_x = tf.transpose(split_x, perm=[0, 2, 1, 3])

#         return split_x

#     def combine_heads(self, x):
#         bsz = x.shape[0]
#         combined_x = tf.transpose(x, perm=[0, 2, 1, 3])
#         combined_x = tf.reshape(combined_x, (bsz, -1, self.d_model))

#         return combined_x


#     def call(self, Q, K, V, mask):
#         WQ = self.W_q(Q)
#         WK = self.W_k(K)
#         WV = self.W_v(V)

#         WQ_splits = self.split_heads(WQ)
#         WK_splits = self.split_heads(WK)
#         WV_splits = self.split_heads(WV)

#         out, attention_weights = self.scaled_dot_product_attention(
#             WQ_splits, WK_splits, WV_splits, mask)

#         out = self.combine_heads(out)
#         out = self.linear(out)

#         return out, attention_weights
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        QK = torch.matmul(Q, K.transpose(-1, -2))
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)
        attentions = F.softmax(scaled_qk, dim=-1)
        out = torch.matmul(attentions, V)
        return out, attentions

    def split_heads(self, x):
        bsz, seq_len, _ = x.size()
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)          # [B, heads, seq, depth]

    def combine_heads(self, x):
        bsz, num_heads, seq_len, depth = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(bsz, seq_len, self.d_model)

    def forward(self, Q, K, V, mask=None):
        WQ, WK, WV = self.W_q(Q), self.W_k(K), self.W_v(V)
        WQ, WK, WV = self.split_heads(WQ), self.split_heads(WK), self.split_heads(WV)
        out, attention_weights = self.scaled_dot_product_attention(WQ, WK, WV, mask)
        out = self.combine_heads(out)
        out = self.linear(out)
        return out, attention_weights
```
</details>


In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

print("슝=3")

슝=3


###### Position-wise Feed Forward Network

In [28]:
# # Position-wise Feed Forward Network 구현
# class PoswiseFeedForwardNet(tf.keras.layers.Layer):
#     def __init__(self, d_model, d_ff):
#         super(PoswiseFeedForwardNet, self).__init__()
#         self.d_model = d_model
#         self.d_ff = d_ff

#         self.fc1 = tf.keras.layers.Dense(d_ff, activation='relu')
#         self.fc2 = tf.keras.layers.Dense(d_model)

#     def call(self, x):
#         out = self.fc1(x)
#         out = self.fc2(out)

#         return out
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))   # Dense(d_ff) + ReLU
        out = self.fc2(out)            # Dense(d_model)
        return out
```
</details>


In [29]:
import torch
import torch.nn as nn

class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        out = self.fc2(out)          # 두 번째 Dense
        return out

print("슝=3")

슝=3


###### Encoder Layer

In [30]:
# # Encoder의 레이어 구현
# class EncoderLayer(tf.keras.layers.Layer):
#     def __init__(self, d_model, n_heads, d_ff, dropout):
#         super(EncoderLayer, self).__init__()

#         self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
#         self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

#         self.norm_1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
#         self.norm_2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

#         self.do = tf.keras.layers.Dropout(dropout)

#     def call(self, x, mask):
#         '''
#         Multi-Head Attention
#         '''
#         residual = x
#         out = self.norm_1(x)
#         out, enc_attn = self.enc_self_attn(out, out, out, mask)
#         out = self.do(out)
#         out += residual

#         '''
#         Position-Wise Feed Forward Network
#         '''
#         residual = out
#         out = self.norm_2(out)
#         out = self.ffn(out)
#         out = self.do(out)
#         out += residual

#         return out, enc_attn
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual
        return out, enc_attn
```
</details>


In [31]:
import torch
import torch.nn as nn

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # nn.LayerNorm은 마지막 차원(d_model)을 기준으로 정규화
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention 단계
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual  # residual connection

        # Position-Wise Feed Forward 단계
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual  # residual connection

        return out, enc_attn

print("슝=3")

슝=3


###### Decoder Layer

In [32]:
# # Decoder 레이어 구현
# class DecoderLayer(tf.keras.layers.Layer):
#     def __init__(self, d_model, num_heads, d_ff, dropout):
#         super(DecoderLayer, self).__init__()

#         self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
#         self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

#         self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

#         self.norm_1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
#         self.norm_2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
#         self.norm_3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

#         self.do = tf.keras.layers.Dropout(dropout)

#     def call(self, x, enc_out, dec_enc_mask, padding_mask):
#         '''
#         Masked Multi-Head Attention
#         '''
#         residual = x
#         out = self.norm_1(x)
#         out, dec_attn = self.dec_self_attn(out, out, out, padding_mask)
#         out = self.do(out)
#         out += residual

#         '''
#         Multi-Head Attention
#         '''
#         residual = out
#         out = self.norm_2(out)
#         # Q, K, V 순서에 주의하세요!
#         out, dec_enc_attn = self.enc_dec_attn(Q=out, K=enc_out, V=enc_out, mask=dec_enc_mask)
#         out = self.do(out)
#         out += residual

#         '''
#         Position-Wise Feed Forward Network
#         '''
#         residual = out
#         out = self.norm_3(out)
#         out = self.ffn(out)
#         out = self.do(out)
#         out += residual

#         return out, dec_attn, dec_enc_attn
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn  = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)
        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # 1) Masked Multi-Head Attention (디코더 자기어텐션)
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # 2) Encoder-Decoder Attention (Q, K, V 순서 주의: Q=디코더, K=V=인코더 출력)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # 3) Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn
```
</details>


In [33]:
import torch
import torch.nn as nn

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn

print("슝=3")

슝=3


###### Encoder

In [34]:
# # Encoder 구현
# class Encoder(tf.keras.Model):
#     def __init__(self,
#                     n_layers,
#                     d_model,
#                     n_heads,
#                     d_ff,
#                     dropout):
#         super(Encoder, self).__init__()
#         self.n_layers = n_layers
#         self.enc_layers = [EncoderLayer(d_model, n_heads, d_ff, dropout)
#                         for _ in range(n_layers)]

#         self.do = tf.keras.layers.Dropout(dropout)

#     def call(self, x, mask):
#         out = x

#         enc_attns = list()
#         for i in range(self.n_layers):
#             out, enc_attn = self.enc_layers[i](out, mask)
#             enc_attns.append(enc_attn)

#         return out, enc_attns
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.n_layers = n_layers
        # 파이썬 list 대신 nn.ModuleList를 써야 파라미터가 제대로 등록됩니다.
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        return out, enc_attns
```
</details>


In [35]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.do = nn.Dropout(dropout)  # 필요 시 입력에 dropout 적용 가능

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        return out, enc_attns

# 사용 예시: Encoder 인스턴스 생성 후 forward 호출
# encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
# out, enc_attns = encoder(x, mask)
print("슝=3")

슝=3


###### Decoder

In [36]:
# # Decoder 구현
# class Decoder(tf.keras.Model):
#     def __init__(self,
#                     n_layers,
#                     d_model,
#                     n_heads,
#                     d_ff,
#                     dropout):
#         super(Decoder, self).__init__()
#         self.n_layers = n_layers
#         self.dec_layers = [DecoderLayer(d_model, n_heads, d_ff, dropout)
#                             for _ in range(n_layers)]

#     def call(self, x, enc_out, dec_enc_mask, padding_mask):
#         out = x

#         dec_attns = list()
#         dec_enc_attns = list()
#         for i in range(self.n_layers):
#             out, dec_attn, dec_enc_attn = \
#             self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)

#             dec_attns.append(dec_attn)
#             dec_enc_attns.append(dec_enc_attn)

#         return out, dec_attns, dec_enc_attns
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns, dec_enc_attns = [], []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](
                out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return out, dec_attns, dec_enc_attns
```
</details>


In [37]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


###### Transformer 전체 모델 조립

In [38]:
# class Transformer(tf.keras.Model):
#     def __init__(self,
#                     n_layers,
#                     d_model,
#                     n_heads,
#                     d_ff,
#                     src_vocab_size,
#                     tgt_vocab_size,
#                     pos_len,
#                     dropout=0.2,
#                     shared_fc=True,
#                     shared_emb=False):
#         super(Transformer, self).__init__()

#         self.d_model = tf.cast(d_model, tf.float32)

#         if shared_emb:
#             self.enc_emb = self.dec_emb = \
#             tf.keras.layers.Embedding(src_vocab_size, d_model)
#         else:
#             self.enc_emb = tf.keras.layers.Embedding(src_vocab_size, d_model)
#             self.dec_emb = tf.keras.layers.Embedding(tgt_vocab_size, d_model)

#         self.pos_encoding = positional_encoding(pos_len, d_model)
#         self.do = tf.keras.layers.Dropout(dropout)

#         self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
#         self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

#         self.fc = tf.keras.layers.Dense(tgt_vocab_size)

#         self.shared_fc = shared_fc

#         if shared_fc:
#             self.fc.set_weights(tf.transpose(self.dec_emb.weights))

#     def embedding(self, emb, x):
#         seq_len = x.shape[1]

#         out = emb(x)

#         if self.shared_fc: out *= tf.math.sqrt(self.d_model)

#         out += self.pos_encoding[np.newaxis, ...][:, :seq_len, :]
#         out = self.do(out)

#         return out


#     def call(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
#         enc_in = self.embedding(self.enc_emb, enc_in)
#         dec_in = self.embedding(self.dec_emb, dec_in)

#         enc_out, enc_attns = self.encoder(enc_in, enc_mask)

#         dec_out, dec_attns, dec_enc_attns = \
#         self.decoder(dec_in, enc_out, dec_enc_mask, dec_mask)

#         logits = self.fc(dec_out)

#         return logits, enc_attns, dec_attns, dec_enc_attns
# print("슝=3")

<details><summary>예시 답안</summary>

```python
class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super().__init__()
        self.d_model = float(d_model)

        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # positional_encoding 결과(넘파이)를 buffer로 등록 (학습 대상 아님)
        pe = positional_encoding(pos_len, d_model)
        self.register_buffer("pos_encoding", torch.tensor(pe, dtype=torch.float32))

        self.do = nn.Dropout(dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # TF의 set_weights(transpose(...)) 대신, PyTorch는 weight를 직접 공유
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        seq_len = x.size(1)
        out = emb(x)
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        return self.do(out)

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)
        enc_out, enc_attns = self.encoder(enc_in, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(
            dec_in, enc_out, dec_enc_mask, dec_mask)
        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns
```
</details>


In [39]:
import torch
import torch.nn as nn
import math

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))

        self.do = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


###### 모델 인스턴스 생성

In [40]:
# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)

transformer = transformer.to(device)

d_model = 512

print("슝=3")

슝=3


<details><summary>예시 답안</summary>

```python
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)

d_model = 512
```
</details>

###### Learning Rate Scheduler

In [41]:
# # Learning Rate Scheduler 구현
# class LearningRateScheduler(tf.keras.optimizers.schedules.LearningRateSchedule):
#     def __init__(self, d_model, warmup_steps=4000):
#         super(LearningRateScheduler, self).__init__()

#         self.d_model = d_model
#         self.warmup_steps = warmup_steps

#     def __call__(self, step):
#         arg1 = step ** -0.5
#         arg2 = step * (self.warmup_steps ** -1.5)

#         return (self.d_model ** -0.5) * tf.math.minimum(arg1, arg2)
# print("슝=3")

In [42]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60): # 4000
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

print("슝=3")

슝=3


<details><summary>예시 답안</summary>

```python
# PyTorch에는 TF의 LearningRateSchedule 베이스 클래스가 없으므로 일반 클래스로 구현하고,
# 학습 루프에서 매 스텝 optimizer의 lr을 직접 갱신합니다.
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)
```
</details>


###### Learning Rate & Optimizer

In [43]:
# # Learning Rate 인스턴스 선언 & Optimizer 구현
# learning_rate = LearningRateScheduler(d_model)

# optimizer = tf.keras.optimizers.Adam(learning_rate,
#                                         beta_1=0.9,
#                                         beta_2=0.98,
#                                         epsilon=1e-9)
# print("슝=3")

<details><summary>예시 답안</summary>

```python
learning_rate = LearningRateScheduler(d_model)

# TF와 달리 스케줄러를 optimizer에 넣을 수 없으므로 초기 lr만 지정하고,
# 학습 루프에서 learning_rate(step) 값으로 param_group['lr']을 갱신합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)
```
</details>


In [44]:
# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)

print("슝=3")

슝=3


###### Loss Function 정의

In [45]:
# # Loss Function 정의
# loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
#     from_logits=True, reduction='none')

# def loss_function(real, pred):
#     mask = tf.math.logical_not(tf.math.equal(real, 0))
#     loss_ = loss_object(real, pred)

#     mask = tf.cast(mask, dtype=loss_.dtype)
#     loss_ *= mask

#     return tf.reduce_sum(loss_)/tf.reduce_sum(mask)
# print("슝=3")

<details><summary>예시 답안</summary>

```python
def loss_function(real, pred):
    # real: [B, T] 정답 인덱스, pred: [B, T, V] logits
    loss_ = F.cross_entropy(
        pred.contiguous().view(-1, pred.size(-1)),
        real.contiguous().view(-1),
        reduction='none')           # from_logits=True, reduction='none'에 해당
    loss_ = loss_.view(real.size())

    mask = (real != 0).float()      # 패딩(0)은 손실 계산에서 제외
    loss_ = loss_ * mask
    return loss_.sum() / mask.sum()
```
</details>


In [46]:
import torch
import torch.nn.functional as F

def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

print("슝=3")

슝=3


###### Train Step 정의

In [47]:
# # Train Step 정의
# @tf.function()
# def train_step(src, tgt, model, optimizer):
#     tgt_in = tgt[:, :-1]  # Decoder의 input
#     gold = tgt[:, 1:]     # Decoder의 output과 비교하기 위해 right shift를 통해 생성한 최종 타겟

#     enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

#     with tf.GradientTape() as tape:
#         predictions, enc_attns, dec_attns, dec_enc_attns = \
#         model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
#         loss = loss_function(gold, predictions)

#     gradients = tape.gradient(loss, model.trainable_variables)
#     optimizer.apply_gradients(zip(gradients, model.trainable_variables))

#     return loss, enc_attns, dec_attns, dec_enc_attns
# print("슝=3")

<details><summary>예시 답안</summary>

```python
def train_step(src, tgt, model, optimizer):
    model.train()
    optimizer.zero_grad()

    tgt_in = tgt[:, :-1]   # Decoder input
    gold   = tgt[:, 1:]    # right-shift한 최종 타겟

    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    predictions, enc_attns, dec_attns, dec_enc_attns = \
        model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
    loss = loss_function(gold, predictions)

    loss.backward()        # tf.GradientTape 대신 autograd
    optimizer.step()       # apply_gradients에 해당
    return loss, enc_attns, dec_attns, dec_enc_attns
```
</details>


In [48]:
def train_step(src, tgt, model, optimizer):
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


###### 훈련을 시키자!

In [49]:
# # Q. 위의 코드를 활용하여 모델을 훈련시켜봅시다!
# EPOCHS = 3

# for epoch in range(EPOCHS):
#     total_loss = 0

#     dataset_count = tf.data.experimental.cardinality(train_dataset).numpy()
#     tqdm_bar = tqdm(total=dataset_count)

#     for (batch, (src, tgt)) in enumerate(train_dataset):
#         loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)
#         total_loss += loss
#         tqdm_bar.set_postfix({"Batch Loss": f"{loss.numpy():.4f}"})
#         tqdm_bar.update(1)

#     tqdm_bar.close()
#     print(f"Epoch {epoch+1}, Loss: {total_loss.numpy() / dataset_count:.4f}")

In [50]:
%%time

TOTAL_EPOCHS = 10 # Set the total number of epochs as requested
START_EPOCH = epoch + 1 if 'epoch' in locals() else 1 # Start from the next epoch if training was interrupted

# global_step is already defined and updated from previous runs
# If starting fresh, uncomment the line below
# global_step = 0

print(f"=== Transformer 챗봇 훈련 재개 (Epoch {START_EPOCH} / {TOTAL_EPOCHS}) ===")
for epoch in range(START_EPOCH, TOTAL_EPOCHS + 1):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    for batch, (src, tgt) in enumerate(train_dataloader):
        # Warmup 스케줄에 맞춰 매 스텝 optimizer의 learning rate를 갱신합니다.
        # (TF에서는 스케줄러를 optimizer에 넣으면 자동 적용되지만, PyTorch에서는 직접 갱신해야 합니다.)
        global_step += 1
        lr = learning_rate(global_step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        # train_step은 cell O7m69dc2nfzm에서 PyTorch 버전으로 정의되었습니다.
        loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)

        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch}/{TOTAL_EPOCHS} - Loss: {total_loss / dataset_count:.4f}")

print("=== Transformer 챗봇 훈련 완료 ===")

=== Transformer 챗봇 훈련 재개 (Epoch 1 / 10) ===


  0%|          | 0/1850 [00:00<?, ?it/s]

NameError: name 'global_step' is not defined

## **17-4. 번역 성능 측정하기 (1) BLEU Score**

멋진 번역 성능 측정 지표인 **BLEU Score**를 기억하시나요? 번역 모델을 훈련한 김에 라이브러리를 활용해서 간단하게 BLEU Score를 실습해 보겠습니다!<br><br>

* [BLEU](https://en.wikipedia.org/wiki/BLEU)

### **NLTK를 활용한 BLEU Score**
---

**NLTK**는 **N**atural **L**anguage **T**ool **K**it 의 준말로 이름부터 자연어 처리에 큰 도움이 될 것 같은 라이브러리입니다.😃 `nltk` 가 BLEU Score를 지원하니 이를 활용하도록 합시다.

In [51]:
# 아래 두 문장을 바꿔가며 테스트 해보세요
reference = "많 은 자연어 처리 연구자 들 이 트랜스포머 를 선호 한다".split()
candidate = "적 은 자연어 학 개발자 들 가 트랜스포머 을 선호 한다 요".split()

print("원문:", reference)
print("번역문:", candidate)
print("BLEU Score:", sentence_bleu([reference], candidate))

원문: ['많', '은', '자연어', '처리', '연구자', '들', '이', '트랜스포머', '를', '선호', '한다']
번역문: ['적', '은', '자연어', '학', '개발자', '들', '가', '트랜스포머', '을', '선호', '한다', '요']
BLEU Score: 8.190757052088229e-155


/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


BLEU Score는 0~1 사이의 값을 가지지만, 100을 곱한 백분율 값으로 표기하는 경우도 많습니다. BLEU Score의 점수대별 해석에 대해서는 [여기](https://cloud.google.com/translate/automl/docs/evaluate?hl=ko#bleu)를 참고해 주세요. \
\
BLEU Score가 **50점을 넘는다는 것은 정말 멋진 번역**을 생성했다는 의미예요. 보통 논문에서 제시하는 BLEU Score는 20점에서 높으면 40점을 바라보는 정도거든요! 하지만 방금 나온 점수는 사실상 0점이라고 해야 하겠네요. 그렇게까지 엉망진창인 번역이 된 것일까요? \
\
BLEU Score의 정의로 돌아가 한번 따져봅시다. BLEU Score가 **N-gram으로 점수를 측정**한다는 것을 기억하실 거예요. 아래 수식을 기억하시죠?

$$
\left( \prod_{i=1}^{4} \text{precision}_i \right)^{\frac{1}{4}} = \left( 1\text{-gram} \times 2\text{-gram} \times 3\text{-gram} \times 4\text{-gram} \right)^{\frac{1}{4}}
$$

**1-gram부터 4-gram까지의 점수(Precision)을 모두 곱한 후, 루트를 두 번 씌우면 $(^\frac{1}{4})$ BLEU Score** 가 된답니다. 진정 멋진 번역이라면, **모든 N-gram에 대해서 높은 점수**를 얻었을 거예요. 그렇다면 위에서 살펴본 예시에서는 각 N-gram이 점수를 얼마나 얻었는지 확인해 보도록 합시다. `weights`의 디폴트 값은 ` [0.25, 0.25, 0.25, 0.25] ` 로 1-gram부터 4-gram까지의 점수에 가중치를 동일하게 주는 것이지만, 만약 이 값을 ` [1, 0, 0, 0] ` 으로 바꿔주면 BLEU Score에 1-gram의 점수만 반영하게 됩니다.


In [52]:
print("1-gram:", sentence_bleu([reference], candidate, weights=[1, 0, 0, 0]))
print("2-gram:", sentence_bleu([reference], candidate, weights=[0, 1, 0, 0]))
print("3-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 1, 0]))
print("4-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 0, 1]))

1-gram: 0.5
2-gram: 0.18181818181818182
3-gram: 2.2250738585072626e-308
4-gram: 2.2250738585072626e-308


0점에 가까운 BLEU Score가 나오는 원인을 알 수 있겠네요. 바로 3-gram와 4-gram에서 거의 0점을 받았기 때문인데요, 위 예시에서 번역문 문장 중 어느 3-gram도 원문의 3-gram과 일치하는 것이 없기 때문입니다. 2-gram이 0.18이 나오는 것은 원문의 11개 2-gram 중에 2개만이 번역문에서 재현되었기 때문입니다. \

하지만 만약 `nltk`의 낮은 버전을 사용할 경우, 간혹 이런 경우에 3-gram, 4-gram 점수가 1이 나와서, 전체적인 BLEU 점수가 50점 이상으로 매우 높게 나오게 될 수도 있습니다.

$$
\left( \prod_{i=1}^{4} \text{precision}_i \right)^{\frac{1}{4}} = \left( 1\text{-gram} \times 2\text{-gram} \times 3\text{-gram} \times 4\text{-gram} \right)^{\frac{1}{4}}
$$

예전 버전에서는 위 수식에서 **어떤 N-gram이 0의 값을 갖는다면 그 하위 N-gram 점수들이 곱했을 때 모두 소멸**해버리기 때문에 일치하는 N-gram이 없더라도 **점수를 `1.0` 으로 유지**하여 **하위 점수를 보존**하게끔 구현되어 있었습니다. 하지만 `1.0` 은 **모든 번역을 완벽히 재현했음을 의미**하기 때문에 총점이 의도치 않게 높아질 수 있어요! 그럴 경우에는 **BLEU Score가 바람직하지 못할 것(Undesirable)** 이라는 경고문이 추가되긴 합니다.

### **`SmoothingFunction()`으로 BLEU Score 보정하기**

그래서 BLEU 계산시 특정 N-gram이 0점이 나와서 BLEU가 너무 커지거나 작아지는 쪽으로 왜곡되는 문제를 보완하기 위해 `SmoothingFunction()` 을 사용하고 있습니다. Smoothing 함수는 **모든 Precision에 아주 작은 `epsilon` 값**을 더해주는 역할을 하는데, 이로써 0점이 부여된 Precision도 완전한 0이 되지 않으니 점수를 `1.0` 으로 대체할 필요가 없어지죠. 즉 **우리의 의도대로 점수가 계산**되는 거예요. \
\
**진실된 BLEU Score**를 확인하기 위해 어서 `SmoothingFunction()` 을 적용해 봅시다! 아래 코드에서는 `SmoothingFunction().method1`을 사용해 보겠습니다. 자신만의 Smoothing 함수를 구현해서 적용할 수도 있겠지만, `nltk`에서는 `method0`부터 `method7`까지를 이미 제공하고 있습니다.<br><br>

* (참고) 각 method들의 상세한 설명은 [nltk의 bleu_score 소스코드](https://www.nltk.org/_modules/nltk/translate/bleu_score.html)를 참고해 봅시다. `sentence_bleu()` 함수에 `smoothing_function=None`을 적용하면 `method0`가 기본 적용됨을 알 수 있습니다.

In [53]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                         candidate,
                         weights=weights,
                         smoothing_function=SmoothingFunction().method1)  # smoothing_function 적용

print("BLEU-1:", calculate_bleu(reference, candidate, weights=[1, 0, 0, 0]))
print("BLEU-2:", calculate_bleu(reference, candidate, weights=[0, 1, 0, 0]))
print("BLEU-3:", calculate_bleu(reference, candidate, weights=[0, 0, 1, 0]))
print("BLEU-4:", calculate_bleu(reference, candidate, weights=[0, 0, 0, 1]))

print("\nBLEU-Total:", calculate_bleu(reference, candidate))

BLEU-1: 0.5
BLEU-2: 0.18181818181818182
BLEU-3: 0.010000000000000004
BLEU-4: 0.011111111111111112

BLEU-Total: 0.05637560315259291


`SmoothingFunction()`로 BLEU score를 보정한 결과, 새로운 BLEU 점수는 무려, 5점으로 올라갔습니다. **거의 의미 없는 번역**이라는 냉정한 평가를 받게 되는군요.😥 \
\
여기서 BLEU-4가 BLEU-3보다 조금이나마 점수가 높은 이유는 **한 문장에서 발생하는 3-gram 쌍의 개수와 4-gram 쌍의 개수**를 생각해 보면 이해할 수 있습니다. **각 Precision을 N-gram 개수로 나누는 부분**에서 차이가 발생하는 것이죠.

### **트랜스포머 모델의 번역 성능 알아보기**
---

위 예시를 조금만 응용하면 우리가 **훈련한 모델이 얼마나 번역을 잘하는지 평가**할 수 있습니다! 아까 **0.5%의 데이터**를 테스트셋으로 빼 둔 것을 기억하시죠? **테스트셋으로 모델의 BLEU Score를 측정**하는 함수 `eval_bleu()` 를 구현해보도록 합시다! \
\
먼저 번역기가 문장을 생성하도록 `translate()` 함수를 정의하겠습니다.

In [54]:
# def translate(tokens, model, src_tokenizer, tgt_tokenizer):
#     padded_tokens = tf.keras.preprocessing.sequence.pad_sequences([tokens],
#                                                            maxlen=MAX_LEN,
#                                                            padding='post')
#     ids = []
#     output = tf.expand_dims([tgt_tokenizer.bos_id()], 0)
#     for i in range(MAX_LEN):
#         enc_padding_mask, combined_mask, dec_padding_mask = \
#         generate_masks(padded_tokens, output)

#         predictions, _, _, _ = model(padded_tokens,
#                                       output,
#                                       enc_padding_mask,
#                                       combined_mask,
#                                       dec_padding_mask)

#         predicted_id = \
#         tf.argmax(tf.math.softmax(predictions, axis=-1)[0, -1]).numpy().item()

#         if tgt_tokenizer.eos_id() == predicted_id:
#             result = tgt_tokenizer.decode_ids(ids)
#             return result

#         ids.append(predicted_id)
#         output = tf.concat([output, tf.expand_dims([predicted_id], 0)], axis=-1)

#     result = tgt_tokenizer.decode_ids(ids)
#     return result

# print("슝=3")

In [55]:
import torch
import torch.nn.functional as F

def translate(tokens, model, src_tokenizer, tgt_tokenizer):
    # tokens: 입력 토큰 리스트
    # MAX_LEN: 최대 길이 (전역 변수 혹은 상수)
    # device: 모델과 데이터가 위치한 디바이스

    # tokens 길이가 MAX_LEN보다 크면 자르고, 작으면 0으로 패딩
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN]
    else:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    # 배치 차원을 추가하여 텐서로 변환 (shape: [1, MAX_LEN])
    padded_tokens = torch.tensor([tokens], dtype=torch.long, device=device)

    ids = []
    # 디코더의 첫 입력은 BOS 토큰 (배치 차원 추가)
    output = torch.tensor([[tgt_tokenizer.bos_id()]], dtype=torch.long, device=device)

    for i in range(MAX_LEN):
        # generate_masks는 padded_tokens와 현재 output으로부터 마스크들을 생성합니다.
        enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(padded_tokens, output)

        # 모델 예측: predictions shape: [batch, seq_len, num_classes]
        predictions, _, _, _ = model(padded_tokens, output, enc_padding_mask, combined_mask, dec_padding_mask)

        # 마지막 시퀀스 위치의 예측값을 소프트맥스 후 argmax로 선택
        predicted_id = predictions[0, -1].softmax(dim=-1).argmax(dim=-1).item()

        # EOS 토큰에 도달하면 현재까지의 예측 토큰 ids를 디코딩 후 반환
        if tgt_tokenizer.eos_id() == predicted_id:
            result = tgt_tokenizer.decode_ids(ids)
            return result

        ids.append(predicted_id)
        # 현재 output에 새로운 예측 토큰을 연결 (dim=1)
        new_token = torch.tensor([[predicted_id]], dtype=torch.long, device=device)
        output = torch.cat([output, new_token], dim=1)

    result = tgt_tokenizer.decode_ids(ids)
    return result

print("슝=3")

슝=3


다음으로 번역한 문장의 BLEU Score를 평가할 수 있도록 함수를 작성합니다. \
\
우선 한 문장만 평가하는 `eval_bleu_single`을 만들어 봅시다.

In [56]:
def eval_bleu_single(model, src_sentence, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    src_tokens = src_tokenizer.encode_as_ids(src_sentence)
    tgt_tokens = tgt_tokenizer.encode_as_ids(tgt_sentence)

    if (len(src_tokens) > MAX_LEN): return None
    if (len(tgt_tokens) > MAX_LEN): return None

    reference = tgt_sentence.split()
    candidate = translate(src_tokens, model, src_tokenizer, tgt_tokenizer).split()

    score = sentence_bleu([reference], candidate,
                          smoothing_function=SmoothingFunction().method1)

    if verbose:
        print("Source Sentence: ", src_sentence)
        print("Model Prediction: ", candidate)
        print("Real: ", reference)
        print("Score: %lf\n" % score)

    return score

print('슝=3')

슝=3


테스트 데이터 중에 하나를 골라 평가해 봅시다.

In [57]:
# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 0

eval_bleu_single(transformer,
                 test_eng_sentences[test_idx],
                 test_spa_sentences[test_idx],
                 tokenizer,
                 tokenizer)

Source Sentence:  when i grow up, i want to be a king.
Model Prediction:  []
Real:  ['quiero', 'ser', 'rey', 'cuando', 'sea', 'grande.']
Score: 0.000000



0

이제 전체 테스트 데이터에 대해서 평가해 봅시다. `eval_bleu_single`을 이용해서 `eval_bleu` 함수를 작성합니다.

In [58]:
def eval_bleu(model, src_sentences, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    total_score = 0.0
    sample_size = len(src_sentences)

    for idx in tqdm(range(sample_size)):
        score = eval_bleu_single(model, src_sentences[idx], tgt_sentence[idx], src_tokenizer, tgt_tokenizer, verbose)
        if not score: continue

        total_score += score

    print("Num of Sample:", sample_size)
    print("Total Score:", total_score / sample_size)

print("슝=3")

슝=3


평가해 봅니다.

In [59]:
eval_bleu(transformer, test_eng_sentences, test_spa_sentences, tokenizer, tokenizer, verbose=False)

  0%|          | 0/594 [00:00<?, ?it/s]

Num of Sample: 594
Total Score: 0.0


## **17-5. 번역 성능 측정하기 (2) Beam Search Decoder**

이 멋진 평가 지표를 더 멋지게 사용하는 방법! 바로 **모델의 생성 기법에 변화를 주는 것**이죠. Greedy Decoding 대신 새로운 기법을 적용하면 **우리 모델을 더 잘 평가할 수 있을 것** 같네요! \
\
**Beam Search**를 기억하나요? 예시로 활용했던 코드를 다시 한번 살펴보면,

In [60]:
def beam_search_decoder(prob, beam_size):
    sequences = [[[], 1.0]]  # 생성된 문장과 점수를 저장

    for tok in prob:
        all_candidates = []

        for seq, score in sequences:
            for idx, p in enumerate(tok): # 각 단어의 확률을 총점에 누적 곱
                candidate = [seq + [idx], score * -math.log(-(p-1))]
                all_candidates.append(candidate)

        ordered = sorted(all_candidates,
                         key=lambda tup:tup[1],
                         reverse=True) # 총점 순 정렬
        sequences = ordered[:beam_size] # Beam Size에 해당하는 문장만 저장

    return sequences

print("슝=3")

슝=3


In [61]:
vocab = {
    0: "<pad>",
    1: "까요?",
    2: "커피",
    3: "마셔",
    4: "가져",
    5: "될",
    6: "를",
    7: "한",
    8: "잔",
    9: "도",
}

prob_seq = [[0.01, 0.01, 0.60, 0.32, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.75, 0.01, 0.01, 0.17],
            [0.01, 0.01, 0.01, 0.35, 0.48, 0.10, 0.01, 0.01, 0.01, 0.01],
            [0.24, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.68],
            [0.01, 0.01, 0.12, 0.01, 0.01, 0.80, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.81, 0.01, 0.01, 0.01, 0.01, 0.11, 0.01, 0.01, 0.01],
            [0.70, 0.22, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]]

prob_seq = np.array(prob_seq)
beam_size = 3

result = beam_search_decoder(prob_seq, beam_size)

for seq, score in result:
    sentence = ""

    for word in seq:
        sentence += vocab[word] + " "

    print(sentence, "// Score: %.4f" % score)

커피 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 42.5243
커피 를 마셔 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 28.0135
마셔 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 17.8983


사실 이 예시는 Beam Search를 설명하는 데에는 더없이 적당하지만 **실제로 모델이 문장을 생성하는 과정과는 거리가 멉니다.** 당장 모델이 문장을 생성하는 과정만 떠올려도 위의 `prob_seq` 처럼 확률을 정의할 수 없겠다는 생각이 머리를 스치죠. 각 단어에 대한 확률은 `prob_seq` 처럼 한 번에 정의가 되지 않고 **이전 스텝까지의 단어에 따라서 결정**되기 때문입니다! \
\
간단한 예시로, Beam Size가 **2**이고 Time-step이 **2**인 순간의 두 문장이 `나는 밥을` , `나는 커피를` 이라고 한다면 세 번째 단어로 `먹는다` , `마신다` 를 고려할 수 있습니다. 이때, 전자에서 `마신다` 에 할당하는 확률과 후자에서 `마신다` 에 할당하는 확률은 **각각 이전 단어들인** `나는 밥을` , `나는 커피를` 에 따라서 결정되기 때문에 **서로 독립적인 확률을 갖습니다.** 예컨대 **후자가 `마신다` 에 더 높은 확률을 할당할 것**을 알 수 있죠! 위 소스에서처럼 "*3번째 단어는 항상* `[마신다: 0.3, 먹는다:0.5, ...]` *의 확률을 가진다!*" 라고는 할 수 없다는 겁니다. \
\
따라서 Beam Search를 생성 기법으로 구현할 때에는 **분기를 잘 나눠줘야 합니다.** Beam Size가 5라고 가정하면 **맨 첫 단어로 적합한 5개의 단어를 생성**하고, 두 번째 단어로 **각 첫 단어(5개 단어)에 대해 5순위**까지 확률을 구하여 **총 25개의 문장을 생성**하죠. 그 25개의 문장들은 각 단어에 할당된 확률을 곱하여 구한 **점수(존재 확률)** 를 가지고 있으니 **각각의 순위**를 매길 수 있겠죠? **점수 상위 5개의 표본**만 살아남아 세 번째 단어를 구할 자격을 얻게 됩니다. \
\
위 과정을 반복하면 최종적으로 점수가 가장 높은 5개의 문장을 얻게 됩니다. 물론 Beam Size를 조절해 주면 그 수는 유동적으로 변할 거구요! 다들 잘 이해하셨죠? 😃

### **Beam Search Decoder 작성 및 평가하기**
---

각 단어의 확률값을 계산하는 `calc_prob()`와 Beam Search를 기반으로 동작하는 `beam_search_decoder()` 를 구현하고 생성된 문장에 대해 BLEU Score를 출력하는 `beam_bleu()` 를 구현하세요! \
\
편의에 따라서 두 기능을 하나의 함수에 구현해도 좋습니다!

In [62]:
# # calc_prob() 구현
# def calc_prob(src_ids, tgt_ids, model):
#     enc_padding_mask, combined_mask, dec_padding_mask = \
#     generate_masks(src_ids, tgt_ids)

#     predictions, enc_attns, dec_attns, dec_enc_attns =\
#     model(src_ids,
#             tgt_ids,
#             enc_padding_mask,
#             combined_mask,
#             dec_padding_mask)

#     return tf.math.softmax(predictions, axis=-1)
# print("슝=3")

<details><summary>예시 코드</summary>

```python
def calc_prob(src_ids, tgt_ids, model):
    enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(src_ids, tgt_ids)
    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src_ids, tgt_ids, enc_padding_mask, combined_mask, dec_padding_mask)
    return F.softmax(predictions, dim=-1)
```
</details>


In [63]:
import torch
import torch.nn.functional as F

def calc_prob(src_ids, tgt_ids, model):
    # 마스크 생성 (PyTorch 버전)
    enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(src_ids, tgt_ids)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src_ids,
        tgt_ids,
        enc_padding_mask,
        combined_mask,
        dec_padding_mask
    )

    # 마지막 차원에 대해 softmax 적용하여 확률값 계산
    return F.softmax(predictions, dim=-1)

print("슝=3")

슝=3


In [64]:
# # beam_search_decoder() 구현
# def beam_search_decoder(sentence,
#                         src_len,
#                         tgt_len,
#                         model,
#                         src_tokenizer,
#                         tgt_tokenizer,
#                         beam_size):
#     tokens = src_tokenizer.encode_as_ids(sentence)

#     src_in = tf.keras.preprocessing.sequence.pad_sequences([tokens],
#                                                             maxlen=src_len,
#                                                             padding='post')

#     pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
#     pred_tmp = np.zeros((beam_size, tgt_len), dtype=np.int64)

#     eos_flag = np.zeros((beam_size, ), dtype=np.int64)
#     scores = np.ones((beam_size, ))

#     pred_tmp[:, 0] = tgt_tokenizer.bos_id()

#     dec_in = tf.expand_dims(pred_tmp[0, :1], 0)
#     prob = calc_prob(src_in, dec_in, model)[0, -1].numpy()

#     for seq_pos in range(1, tgt_len):
#         score_cache = np.ones((beam_size * beam_size, ))

#         # init
#         for branch_idx in range(beam_size):
#             cache_pos = branch_idx*beam_size

#             score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
#             pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = \
#             pred_tmp[branch_idx, :seq_pos]

#         for branch_idx in range(beam_size):
#             cache_pos = branch_idx*beam_size

#             if seq_pos != 1:   # 모든 Branch를 로 시작하는 경우를 방지
#                 dec_in = pred_cache[branch_idx, :seq_pos]
#                 dec_in = tf.expand_dims(dec_in, 0)

#                 prob = calc_prob(src_in, dec_in, model)[0, -1].numpy()

#             for beam_idx in range(beam_size):
#                 max_idx = np.argmax(prob)

#                 score_cache[cache_pos+beam_idx] *= prob[max_idx]
#                 pred_cache[cache_pos+beam_idx, seq_pos] = max_idx

#                 prob[max_idx] = -1

#         for beam_idx in range(beam_size):
#             if eos_flag[beam_idx] == -1: continue

#             max_idx = np.argmax(score_cache)
#             prediction = pred_cache[max_idx, :seq_pos+1]

#             pred_tmp[beam_idx, :seq_pos+1] = prediction
#             scores[beam_idx] = score_cache[max_idx]
#             score_cache[max_idx] = -1

#             if prediction[-1] == tgt_tokenizer.eos_id():
#                 eos_flag[beam_idx] = -1

#     pred = []
#     for long_pred in pred_tmp:
#         zero_idx = long_pred.tolist().index(tgt_tokenizer.eos_id())
#         short_pred = long_pred[:zero_idx+1]
#         pred.append(short_pred)
#     return pred
# print("슝=3")

<details><summary>예시 코드</summary>

```python
def beam_search_decoder(sentence, src_len, tgt_len, model,
                        src_tokenizer, tgt_tokenizer, beam_size):
    tokens = src_tokenizer.encode_as_ids(sentence)

    # tf.keras...pad_sequences 대신 numpy로 직접 padding 후 텐서로 변환
    padded = np.zeros((1, src_len), dtype=np.int64)
    padded[0, :len(tokens)] = tokens
    src_in = torch.tensor(padded, dtype=torch.long, device=device)

    pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
    pred_tmp   = np.zeros((beam_size, tgt_len), dtype=np.int64)
    eos_flag   = np.zeros((beam_size,), dtype=np.int64)
    scores     = np.ones((beam_size,), dtype=np.float32)

    pred_tmp[:, 0] = tgt_tokenizer.bos_id()

    dec_in = torch.tensor(pred_tmp[0, :1], dtype=torch.long, device=device).unsqueeze(0)
    prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

    for seq_pos in range(1, tgt_len):
        score_cache = np.ones((beam_size * beam_size,), dtype=np.float32)

        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
            pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = pred_tmp[branch_idx, :seq_pos]

        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            if seq_pos != 1:
                dec_in = torch.tensor(pred_cache[branch_idx, :seq_pos],
                                      dtype=torch.long, device=device).unsqueeze(0)
                prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

            for beam_idx in range(beam_size):
                max_idx = np.argmax(prob)
                score_cache[cache_pos + beam_idx] *= prob[max_idx]
                pred_cache[cache_pos + beam_idx, seq_pos] = max_idx
                prob[max_idx] = -1

        for beam_idx in range(beam_size):
            if eos_flag[beam_idx] == -1:
                continue
            max_idx = np.argmax(score_cache)
            prediction = pred_cache[max_idx, :seq_pos+1].copy()
            pred_tmp[beam_idx, :seq_pos+1] = prediction
            scores[beam_idx] = score_cache[max_idx]
            score_cache[max_idx] = -1
            if prediction[-1] == tgt_tokenizer.eos_id():
                eos_flag[beam_idx] = -1

    pred = []
    for long_pred in pred_tmp:
        try:
            eos_idx = list(long_pred).index(tgt_tokenizer.eos_id())
        except ValueError:
            eos_idx = tgt_len - 1          # EOS가 없으면 전체 시퀀스 사용
        pred.append(long_pred[:eos_idx+1].tolist())
    return pred
```
</details>


In [65]:
import numpy as np
import torch

def beam_search_decoder(sentence,
                        src_len,
                        tgt_len,
                        model,
                        src_tokenizer,
                        tgt_tokenizer,
                        beam_size):
    # 입력 문장을 토큰화
    tokens = src_tokenizer.encode_as_ids(sentence)

    # src_in: [1, src_len] 크기의 텐서로 padding (0: 패딩 토큰)
    padded = np.zeros((1, src_len), dtype=np.int64)
    padded[0, :len(tokens)] = tokens
    src_in = torch.tensor(padded, dtype=torch.long, device=device)

    # beam search용 캐시 배열들
    pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
    pred_tmp = np.zeros((beam_size, tgt_len), dtype=np.int64)

    eos_flag = np.zeros((beam_size,), dtype=np.int64)  # EOS를 만난 branch 표시 (EOS: -1)
    scores = np.ones((beam_size,), dtype=np.float32)     # 각 branch의 score (확률 곱)

    # 디코더 첫 입력은 BOS 토큰
    pred_tmp[:, 0] = tgt_tokenizer.bos_id()

    # 초기 디코더 입력 (branch 0의 첫 토큰) -> shape: [1, 1]
    dec_in = torch.tensor(pred_tmp[0, :1], dtype=torch.long, device=device).unsqueeze(0)
    # calc_prob()는 softmax를 적용한 확률 텐서를 반환함
    prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

    # seq_pos: 디코더 시퀀스 위치
    for seq_pos in range(1, tgt_len):
        score_cache = np.ones((beam_size * beam_size,), dtype=np.float32)

        # 각 beam branch에 대해 캐시 초기화
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
            pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = pred_tmp[branch_idx, :seq_pos]

        # 각 beam branch에 대해 후보 확률 계산 및 캐시 업데이트
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            if seq_pos != 1:
                # 해당 branch의 현재까지의 시퀀스를 디코더 입력으로 변환
                dec_in_np = pred_cache[branch_idx, :seq_pos]
                dec_in = torch.tensor(dec_in_np, dtype=torch.long, device=device).unsqueeze(0)
                prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

            # 각 branch 내에서 beam_size만큼의 후보 토큰을 선택
            for beam_idx in range(beam_size):
                max_idx = np.argmax(prob)
                # 후보 branch의 score 업데이트 (곱셈으로 누적)
                score_cache[cache_pos + beam_idx] *= prob[max_idx]
                pred_cache[cache_pos + beam_idx, seq_pos] = max_idx
                # 이미 선택된 토큰은 다시 선택되지 않도록 -1로 마킹
                prob[max_idx] = -1

        # 각 beam branch에서 최고 score를 가진 후보를 선택
        for beam_idx in range(beam_size):
            if eos_flag[beam_idx] == -1:
                continue
            max_idx = np.argmax(score_cache)
            prediction = pred_cache[max_idx, :seq_pos+1].copy()
            pred_tmp[beam_idx, :seq_pos+1] = prediction
            scores[beam_idx] = score_cache[max_idx]
            score_cache[max_idx] = -1  # 해당 후보 제거

            # 만약 EOS 토큰이면 해당 branch는 종료 표시 (-1)
            if prediction[-1] == tgt_tokenizer.eos_id():
                eos_flag[beam_idx] = -1

    # 각 branch의 예측 시퀀스에서 EOS 토큰 이전까지만 추출하여 결과 반환
    pred = []
    for long_pred in pred_tmp:
        eos_token = tgt_tokenizer.eos_id()
        # EOS 토큰이 없는 경우, 전체 시퀀스를 사용하도록 처리할 수 있음
        try:
            eos_idx = list(long_pred).index(eos_token)
        except ValueError:
            eos_idx = tgt_len - 1
        short_pred = long_pred[:eos_idx+1]
        pred.append(short_pred.tolist())

    return pred

print("슝=3")

슝=3


BLEU 계산 함수는 이전과 동일하게 사용할게요.

In [66]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                            candidate,
                            weights=weights,
                            smoothing_function=SmoothingFunction().method1)

print('슝=3')

슝=3


마지막으로 `beam_bleu`함수를 만들어 주세요.

In [67]:
# # Q. beam_bleu() 함수를 구현해봅시다.
# def beam_bleu(reference, ids, tokenizer):
#     reference = reference.split()

#     total_score = 0.0
#     for _id in ids:
#         candidate = tokenizer.decode_ids(_id.tolist()).split()
#         score = calculate_bleu(reference, candidate)

#         print("Reference:", reference)
#         print("Candidate:", candidate)
#         print("BLEU:", calculate_bleu(reference, candidate))

#         total_score += score

#     return total_score / len(ids)
# print("슝=3")

<details><summary>예시 코드</summary>

```python
def beam_bleu(reference, ids, tokenizer):
    reference = reference.split()

    total_score = 0.0
    for _id in ids:
        candidate = tokenizer.decode_ids(_id.tolist()).split()
        score = calculate_bleu(reference, candidate)

        print("Reference:", reference)
        print("Candidate:", candidate)
        print("BLEU:", calculate_bleu(reference, candidate))

        total_score += score
        
    return total_score / len(ids)
```
</details>

In [68]:
def beam_bleu(reference, ids, tokenizer):
    # 기준 문장을 토큰화
    reference_tokens = reference.split()

    total_score = 0.0
    num_candidates = len(ids)
    if num_candidates == 0:
        return 0.0

    for candidate_ids in ids:
        # 후보 문장을 디코딩 후 토큰화
        candidate_sentence = tokenizer.decode_ids(candidate_ids)
        candidate_tokens = candidate_sentence.split()

        score = calculate_bleu(reference_tokens, candidate_tokens)

        print(f"Reference: {reference_tokens}")
        print(f"Candidate: {candidate_tokens}")
        print(f"BLEU: {score}")

        total_score += score

    return total_score / num_candidates

print("슝=3")

슝=3


구현 후 다음과 같이 사용합니다.

In [69]:
# Q. 인덱스를 바꿔가며 확인해 보세요
test_idx = 1

ids = \
beam_search_decoder(test_eng_sentences[test_idx],
                    MAX_LEN,
                    MAX_LEN,
                    transformer,
                    tokenizer,
                    tokenizer,
                    beam_size=5)

bleu = beam_bleu(test_spa_sentences[test_idx], ids, tokenizer)
print(bleu)

Reference: ['¿puedes', 'esperar', 'un', 'poco', 'más?']
Candidate: []
BLEU: 0
Reference: ['¿puedes', 'esperar', 'un', 'poco', 'más?']
Candidate: []
BLEU: 0
Reference: ['¿puedes', 'esperar', 'un', 'poco', 'más?']
Candidate: []
BLEU: 0
Reference: ['¿puedes', 'esperar', 'un', 'poco', 'más?']
Candidate: []
BLEU: 0
Reference: ['¿puedes', 'esperar', 'un', 'poco', 'más?']
Candidate: []
BLEU: 0
0.0


## **17-6. 데이터 부풀리기**

이번 스텝에서는 **Data Augmentation**, 그중에서도 **Embedding을 활용한 Lexical Substitution**을 구현해 볼 거예요. `gensim` 라이브러리를 활용하면 어렵지 않게 해낼 수 있습니다! \
\
`gensim` 에 사전 훈련된 Embedding 모델을 불러오는 것은 두 가지 방법이 있습니다.<br><br>

1. **직접 모델을 다운로드해 `load`** 하는 방법
2. `gensim` 이 자체적으로 지원하는 **`downloader` 를 활용해 모델을 `load`** 하는 방법

<br>한국어는 `gensim` 에서 지원하지 않으므로 두 번째 방법을 사용할 수 없지만, **영어라면 얘기가 달라지죠!** 아래 웹페이지의 `Available data → Model` 부분에서 공개된 모델의 종류를 확인할 수 있습니다.<br><br>

* [RaRe-Technologies/gensim-data](https://github.com/RaRe-Technologies/gensim-data)

<br>대표적으로 사용되는 Embedding 모델은 `word2vec-google-news-300` 이지만 용량이 커서 다운로드에 많은 시간이 소요되므로 이번 실습엔 적합하지 않습니다. 우리는 적당한 사이즈의 모델인 `glove-wiki-gigaword-300` 을 사용할게요! 아래 소스를 실행해 **사전 훈련된 Embedding 모델을 다운로드**해 주세요.

In [70]:
import gensim.downloader as api

wv = api.load('glove-wiki-gigaword-300')

[==================================================] 100.0% 376.1/376.1MB downloaded


불러온 모델은 아래와 같이 활용할 수 있습니다.

In [71]:
wv.most_similar("banana")

[('bananas', 0.6691170930862427),
 ('mango', 0.5804104208946228),
 ('pineapple', 0.5492372512817383),
 ('coconut', 0.5462778806686401),
 ('papaya', 0.541056752204895),
 ('fruit', 0.52181077003479),
 ('growers', 0.4877638816833496),
 ('nut', 0.48399588465690613),
 ('peanut', 0.48062023520469666),
 ('potato', 0.48061180114746094)]

주어진 데이터를 토큰 단위로 분리한 후, 랜덤하게 하나를 선정하여 해당 토큰과 가장 유사한 단어를 찾아 대치하면 그것으로 **Lexical Substitution**은 완성되겠죠? 가볍게 확인해 봅시다!

In [72]:
sample_sentence = "you know ? all you need is attention ."
sample_tokens = sample_sentence.split()

selected_tok = random.choice(sample_tokens)

result = ""
for tok in sample_tokens:
    if tok == selected_tok:
        result += wv.most_similar(tok)[0][0] + " "

    else:
        result += tok + " "

print("From:", sample_sentence)
print("To:", result)

From: you know ? all you need is attention .
To: you know ? all you need is focus . 


### **Lexical Substitution 구현하기**
---

입력된 문장을 Embedding 유사도를 기반으로 Augmentation 하여 반환하는 `lexical_sub()` 를 구현하세요!

In [73]:
# Q. Lexical Substitution 을 구현해봅시다.
def lexical_sub(sentence, wv):
    # 문장을 토큰화
    tokens = sentence.split()

    # 유효한 단어 필터링 (임베딩에 존재하는 단어만 고려)
    valid_tokens = [tok for tok in tokens if tok in wv]

    # 대체할 단어 선택 (임베딩 내 존재하는 단어 중 하나)
    if not valid_tokens:
        return sentence  # 모든 단어가 임베딩 내에 없으면 원래 문장 반환

    selected_tok = random.choice(valid_tokens)

    # 가장 유사한 단어 찾기
    similar_word = wv.most_similar(selected_tok)[0][0]

    # 변환된 문장 생성
    new_sentence = " ".join([similar_word if tok == selected_tok else tok for tok in tokens])

    return new_sentence

만들어진 함수를 사용해 볼까요? 시간이 오래 걸리니 우선 테스트 데이터의 Augmentation들을 만들어 봅시다. 약간 시간이 걸립니다. 프로젝트에서는 학습 데이터로 Augmentation을 해야합니다~!

In [74]:
new_corpus = []

for old_src in tqdm(test_eng_sentences):
    new_src = lexical_sub(old_src, wv)
    if new_src is not None:
        new_corpus.append(new_src)
    # Augmentation이 없더라도 원본 문장을 포함시킵니다
    new_corpus.append(old_src)

print(new_corpus[:10])

  0%|          | 0/594 [00:00<?, ?it/s]

['then i grow up, i want to be a king.', 'when i grow up, i want to be a king.', "can 'll hold on a little longer?", 'can you hold on a little longer?', "'d can peel an apple.", 'i can peel an apple.', 'these you waiting for someone?', 'are you waiting for someone?', 'no another uses that word anymore.', 'no one uses that word anymore.']


# **18. 번역가는 대화에도 능하다 [프로젝트]**

## **18-1. Project: 멋진 챗봇 만들기**

### **라이브러리 버전을 확인해 봅니다**
---

사용할 라이브러리 버전을 둘러봅시다.

In [75]:
import numpy
import pandas
import torch
import nltk
import gensim

print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.1.3
2.2.3
2.11.0+cu128
3.9.1
4.4.0


지난 노드에서 **챗봇과 번역기는 같은 집안**이라고 했던 말을 기억하시나요? \
앞서 배운 Seq2seq번역기와 Transfomer번역기에 적용할 수도 있겠지만, 이번 노드에서 배운 번역기 성능 측정법을 챗봇에도 적용해 봅시다. 배운 지식을 다양하게 활용할 수 있는 것도 중요한 능력이겠죠. 이번 프로젝트를 통해서 챗봇과 번역기가 같은 집안인지 확인해 보세요!

### **Step 1. 데이터 다운로드**
---

준비하기 단계에서 심볼릭 링크를 생성했다면 아래 파일이 `ChatbotData.csv`라는 이름으로 저장되어 있을거예요. `csv` 파일을 읽는 데에는 `pandas` 라이브러리가 적합합니다. 읽어 온 데이터의 질문과 답변을 각각 `questions`, `answers` 변수에 나눠서 저장하세요!

* [songys/Chatbot_data](https://github.com/songys/Chatbot_data)

### **Step 2. 데이터 정제**
---

아래 조건을 만족하는 `preprocess_sentence()` 함수를 구현하세요.<br><br>

1. 영문자의 경우, **모두 소문자**로 변환합니다.
2. 영문자와 한글, 숫자, 그리고 주요 특수문자를 제외하곤 **정규식을 활용하여 모두 제거**합니다.

<br>*문장부호 양옆에 공백을 추가하는 등 이전과 다르게 생략된 기능들은 우리가 사용할 토크나이저가 지원하기 때문에 굳이 구현하지 않아도 괜찮습니다!*

### **Step 3. 데이터 토큰화**
---

토큰화에는 *KoNLPy*의 `mecab` 클래스를 사용합니다. \
\
아래 조건을 만족하는 `build_corpus()` 함수를 구현하세요!<br><br>

1. **소스 문장 데이터**와 **타겟 문장 데이터**를 입력으로 받습니다.
2. 데이터를 앞서 정의한 **`preprocess_sentence()`** 함수로 **정제하고, 토큰화**합니다.
3. 토큰화는 **전달받은 토크나이즈 함수를 사용**합니다. 이번엔 **`mecab.morphs`** 함수를 전달하시면 됩니다.
4. 토큰의 개수가 일정 길이 이상인 문장은 **데이터에서 제외**합니다.
5. **중복되는 문장은 데이터에서 제외**합니다. `소스 : 타겟` 쌍을 비교하지 않고 소스는 소스대로 타겟은 타겟대로 검사합니다. 중복 쌍이 흐트러지지 않도록 유의하세요!

<br>구현한 함수를 활용하여 `questions` 와 `answers` 를 각각 `que_corpus` , `ans_corpus` 에 토큰화하여 저장합니다.

In [76]:
"""
Step 1: 데이터 다운로드 및 불러오기
Step 2: 데이터 정제 (preprocess_sentence)
Step 3: 데이터 토큰화 및 중복/길이 필터링 (build_corpus)
"""

import os
import re
import pandas as pd
from konlpy.tag import Mecab

# 1. 데이터 로드 (Step 1)
data_path = 'ChatbotData.csv'
if not os.path.exists(data_path):
    # 파일이 없을 경우 다운로드
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv", filename=data_path)

df = pd.read_csv(data_path)
questions = df['Q'].tolist()
answers = df['A'].tolist()

print(f"원 데이터 개수: 질문 {len(questions)}개, 답변 {len(answers)}개")

# 2. 데이터 정제 함수 정의 (Step 2)
def preprocess_sentence(sentence: str) -> str:
    """
    [루브릭 1 연계 - 데이터 정제]
    1. 영문 소문자 변환
    2. 영문, 한글, 숫자, 주요 특수문자(?, !, ., ,)를 제외한 기호 정규식 제거
    """
    sentence = sentence.lower().strip()
    sentence = re.sub(r"[^a-zA-a-가-힣0-9?.!,]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip()

# 3. 데이터 토큰화 및 Corpus 구축 함수 정의 (Step 3)
mecab = Mecab()

def build_corpus(questions, answers, tokenizer_func=mecab.morphs, max_len=20):
    """
    [루브릭 1 연계 - 토큰화 및 중복/길이 제거]
    - 정제 및 mecab 토큰화 수행
    - max_len 초과 문장 제외
    - 소스(Q), 타겟(A) 각각 중복 제외 처리
    """
    que_corpus = []
    ans_corpus = []

    seen_q = set()
    seen_a = set()

    for q, a in zip(questions, answers):
        q_clean = preprocess_sentence(q)
        a_clean = preprocess_sentence(a)

        q_tokens = tokenizer_func(q_clean)
        a_tokens = tokenizer_func(a_clean)

        # 길이 조건 체크
        if len(q_tokens) > max_len or len(a_tokens) > max_len:
            continue

        # 중복 체크 (Q 중복 또는 A 중복 시 제외)
        q_str = " ".join(q_tokens)
        a_str = " ".join(a_tokens)

        if q_str in seen_q or a_str in seen_a:
            continue

        seen_q.add(q_str)
        seen_a.add(a_str)

        que_corpus.append(q_tokens)
        ans_corpus.append(a_tokens)

    return que_corpus, ans_corpus

que_corpus, ans_corpus = build_corpus(questions, answers, tokenizer_func=mecab.morphs, max_len=20)
print(f"정제 및 중복/길이 필터링 후 데이터 개수: {len(que_corpus)}개")

원 데이터 개수: 질문 11823개, 답변 11823개


Exception: The MeCab dictionary does not exist at "/usr/local/lib/mecab/dic/mecab-ko-dic". Is the dictionary correctly installed?
You can also try entering the dictionary path when initializing the Mecab class: "Mecab('/some/dic/path')"

### **Step 4. Augmentation**
---
`Kyubyong/wordvectors`의 **Korean (w)** Word2Vec 모델(`ko.bin`)을 **자동으로 다운로드**하여
Lexical Substitution으로 데이터를 약 3배로 늘립니다. (원본 노트북에서 수동 업로드를 요구하던 부분을 자동화했습니다.)

In [ ]:
!pip install -q gdown

import os, zipfile, gdown

# Kyubyong/wordvectors README의 'Korean (w)' Google Drive 파일 ID
KO_W2V_GDRIVE_ID = "0B0ZXk88koS2KbDhXdWg1Q2RydlU"
KO_ZIP_PATH = "ko.zip"
KO_BIN_PATH = "ko.bin"

if not os.path.exists(KO_BIN_PATH):
    if not os.path.exists(KO_ZIP_PATH):
        gdown.download(id=KO_W2V_GDRIVE_ID, output=KO_ZIP_PATH, quiet=False)

    with zipfile.ZipFile(KO_ZIP_PATH, "r") as zf:
        zf.extractall(".")
        print("압축 해제된 파일들:", zf.namelist())

print("ko.bin 존재 여부:", os.path.exists(KO_BIN_PATH))


In [ ]:
import random
from gensim.models import Word2Vec

word2vec = None
try:
    word2vec = Word2Vec.load(KO_BIN_PATH)
    print("Word2Vec 모델 로드 완료! (vocab size:", len(word2vec.wv), ")")
except Exception as e:
    print(f"[경고] Word2Vec 모델 로드 실패: {e}")
    print("데이터 증강(Augmentation)이 건너뛰어집니다.")


def lexical_sub(tokens, word2vec_model, sub_ratio=0.2):
    """토큰 중 일부를 Word2Vec 유사 단어로 무작위 치환"""
    if word2vec_model is None:
        return tokens

    new_tokens = tokens.copy()
    num_to_permute = max(1, int(len(tokens) * sub_ratio))

    candidate_indices = [i for i, tok in enumerate(tokens) if tok in word2vec_model.wv]
    if not candidate_indices:
        return tokens

    selected_indices = random.sample(candidate_indices, min(len(candidate_indices), num_to_permute))

    for idx in selected_indices:
        target_word = tokens[idx]
        similar_words = word2vec_model.wv.most_similar(target_word, topn=5)
        if similar_words:
            new_tokens[idx] = random.choice(similar_words)[0]

    return new_tokens


def augment_corpus(que_corpus, ans_corpus, word2vec_model):
    """
    원본 + (Augment Q + Original A) + (Original Q + Augment A) = 약 3배 데이터 생성
    (버그 수정: 두 번째 루프에서 실제로 증강된 a_aug를 사용하도록 고침)
    """
    augmented_que = list(que_corpus)
    augmented_ans = list(ans_corpus)

    if word2vec_model is None:
        return augmented_que, augmented_ans

    # 1. Augment Q + Original A
    for q_tokens, a_tokens in zip(que_corpus, ans_corpus):
        q_aug = lexical_sub(q_tokens, word2vec_model)
        augmented_que.append(q_aug)
        augmented_ans.append(a_tokens)

    # 2. Original Q + Augment A  (기존 버그: a_aug 대신 a_tokens가 들어가던 부분을 수정)
    for q_tokens, a_tokens in zip(que_corpus, ans_corpus):
        a_aug = lexical_sub(a_tokens, word2vec_model)
        augmented_que.append(q_tokens)
        augmented_ans.append(a_aug)

    return augmented_que, augmented_ans


if word2vec is not None:
    que_corpus_total, ans_corpus_total = augment_corpus(que_corpus, ans_corpus, word2vec)
    print(f"=== 데이터 Augmentation 완료 ===")
    print(f"최종 학습 데이터 수: {len(que_corpus_total)}개 (원본 {len(que_corpus)}개 대비 약 {len(que_corpus_total)/len(que_corpus):.1f}배)")
else:
    que_corpus_total, ans_corpus_total = list(que_corpus), list(ans_corpus)
    print("Word2Vec 로드 실패로 원본 데이터만 사용합니다.")


### **Step 5. 데이터 벡터화 (PyTorch)**
---
⚠️ 원본 노트북은 여기서 `tensorflow`(`tf.keras.preprocessing.text.Tokenizer`)로 전환하면서
17장에서 만든 PyTorch Transformer와 완전히 어긋나 버렸습니다. **여기서부터는 순수 PyTorch로만 진행합니다.**

In [ ]:
# 1. 타겟(답변) 데이터에 <start>, <end> 추가
ans_corpus_tagged = [["<start>"] + toks + ["<end>"] for toks in ans_corpus_total]
que_corpus_tagged = que_corpus_total

# 2. 공유 단어 사전 구축 (질문/답변이 같은 언어이므로 Embedding 공유에 유리)
from collections import Counter

PAD, START, END, UNK = "<pad>", "<start>", "<end>", "<unk>"
SPECIAL_TOKENS = [PAD, START, END, UNK]

def build_vocab(corpus_a, corpus_b, min_freq=1):
    counter = Counter()
    for tokens in corpus_a + corpus_b:
        counter.update(tokens)

    vocab = list(SPECIAL_TOKENS)
    for tok, freq in counter.most_common():
        if tok in SPECIAL_TOKENS:
            continue
        if freq >= min_freq:
            vocab.append(tok)

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word


word2idx, idx2word = build_vocab(que_corpus_tagged, ans_corpus_tagged, min_freq=1)
VOCAB_SIZE = len(word2idx)
print("VOCAB_SIZE:", VOCAB_SIZE)


def tokens_to_ids(tokens, word2idx):
    return [word2idx.get(tok, word2idx[UNK]) for tok in tokens]


def pad_sequences_custom(sequences, max_len, pad_value=0):
    padded = []
    for seq in sequences:
        seq = seq[:max_len] if len(seq) > max_len else seq + [pad_value] * (max_len - len(seq))
        padded.append(seq)
    return torch.tensor(padded, dtype=torch.long)


MAX_LEN = 22  # <start>, <end> 포함 고려

que_ids = [tokens_to_ids(toks, word2idx) for toks in que_corpus_tagged]
ans_ids = [tokens_to_ids(toks, word2idx) for toks in ans_corpus_tagged]

enc_train = pad_sequences_custom(que_ids, max_len=MAX_LEN, pad_value=word2idx[PAD])
dec_train = pad_sequences_custom(ans_ids, max_len=MAX_LEN, pad_value=word2idx[PAD])

print("enc_train:", enc_train.shape, " dec_train:", dec_train.shape)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 학습/테스트 분리 (Step 7 BLEU 측정을 위해 필요)
enc_np, dec_np = enc_train.numpy(), dec_train.numpy()
enc_tr, enc_te, dec_tr, dec_te = train_test_split(enc_np, dec_np, test_size=0.05, random_state=42)

enc_tr = torch.tensor(enc_tr, dtype=torch.long)
dec_tr = torch.tensor(dec_tr, dtype=torch.long)
enc_te = torch.tensor(enc_te, dtype=torch.long)
dec_te = torch.tensor(dec_te, dtype=torch.long)

CHATBOT_BATCH_SIZE = 64
chatbot_train_dataset = TensorDataset(enc_tr, dec_tr)
chatbot_train_dataloader = DataLoader(chatbot_train_dataset, batch_size=CHATBOT_BATCH_SIZE, shuffle=True, pin_memory=True)

print("Train:", enc_tr.shape, " Test:", enc_te.shape)


### **Step 6. 훈련하기 (17장의 PyTorch Transformer 재사용)**
---
⚠️ 원본 노트북은 여기서 `tf.keras.Model` 기반 Transformer를 처음부터 새로 구현해서 17장에서 만든 모델을 버렸습니다.
**여기서는 17장에서 이미 정의된 `Transformer`, `generate_masks`, `train_step`, `LearningRateScheduler`, `loss_function`을 그대로 재사용**합니다.
데이터가 3만 개 내외로 작으므로 과적합 방지를 위해 레이어를 줄이고 dropout을 높인 하이퍼파라미터를 기본값으로 두되,
**실제로는 train/val loss 추이를 보고 직접 조정하는 것을 권장**합니다.

In [ ]:
# 챗봇 전용 하이퍼파라미터 (과적합 방지를 위해 17장보다 작게 설정 — 출발점일 뿐, 관찰 후 조정 권장)
CHAT_N_LAYERS = 1
CHAT_D_MODEL = 368
CHAT_N_HEADS = 8
CHAT_D_FF = 1024
CHAT_DROPOUT = 0.2
CHAT_WARMUP_STEPS = 1000
CHAT_EPOCHS = 10

# 17장에서 정의된 Transformer 클래스를 그대로 재사용 (새로 정의하지 않음!)
chatbot_transformer = Transformer(
    n_layers=CHAT_N_LAYERS,
    d_model=CHAT_D_MODEL,
    n_heads=CHAT_N_HEADS,
    d_ff=CHAT_D_FF,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=MAX_LEN + 1,
    dropout=CHAT_DROPOUT,
    shared_fc=True,
    shared_emb=True,   # 질문/답변이 같은 언어이므로 임베딩 공유
).to(device)

chat_d_model = CHAT_D_MODEL
chat_learning_rate = LearningRateScheduler(chat_d_model, warmup_steps=CHAT_WARMUP_STEPS)
chat_optimizer = torch.optim.Adam(chatbot_transformer.parameters(),
                                   lr=chat_learning_rate(1), betas=(0.9, 0.98), eps=1e-9)
print("슝=3")


In [ ]:
@torch.no_grad()
def eval_loss_chatbot(model, enc_data, dec_data, batch_size=64):
    model.eval()
    ds = TensorDataset(enc_data, dec_data)
    dl = DataLoader(ds, batch_size=batch_size)
    total_loss, n_batches = 0.0, 0
    for src, tgt in dl:
        tgt_in, gold = tgt[:, :-1], tgt[:, 1:]
        enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)
        src, tgt_in = src.to(device), tgt_in.to(device)
        enc_mask, dec_enc_mask, dec_mask = enc_mask.to(device), dec_enc_mask.to(device), dec_mask.to(device)
        predictions, *_ = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)
        total_loss += loss_function(gold, predictions).item()
        n_batches += 1
    return total_loss / n_batches


chat_global_step = 0
chat_train_hist, chat_val_hist = [], []

print("=== 챗봇 Transformer 훈련 시작 ===")
for ep in range(CHAT_EPOCHS):
    total_loss = 0.0
    dataset_count = len(chatbot_train_dataloader)
    tqdm_bar = tqdm(total=dataset_count)

    for src, tgt in chatbot_train_dataloader:
        chat_global_step += 1
        lr = chat_learning_rate(chat_global_step)
        for pg in chat_optimizer.param_groups:
            pg['lr'] = lr

        # train_step은 17장에서 이미 정의된 PyTorch 버전을 그대로 사용
        loss, *_ = train_step(src, tgt, chatbot_transformer, chat_optimizer)
        total_loss += loss.item()
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    train_avg = total_loss / dataset_count
    val_avg = eval_loss_chatbot(chatbot_transformer, enc_te, dec_te)
    chat_train_hist.append(train_avg)
    chat_val_hist.append(val_avg)
    print(f"Epoch {ep+1}/{CHAT_EPOCHS}  train_loss={train_avg:.4f}  val_loss={val_avg:.4f}")

print("=== 챗봇 Transformer 훈련 완료 ===")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(chat_train_hist, label="train_loss")
plt.plot(chat_val_hist, label="val_loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Chatbot Train vs Val Loss (과적합 여부 확인)")
plt.show()


In [ ]:
def translate_chatbot(model, sentence, word2idx, idx2word, max_len=MAX_LEN):
    tokens = mecab.morphs(preprocess_sentence(sentence))
    ids = tokens_to_ids(tokens, word2idx)
    ids = ids[:max_len] if len(ids) > max_len else ids + [word2idx[PAD]] * (max_len - len(ids))

    src_in = torch.tensor([ids], dtype=torch.long, device=device)
    output = torch.tensor([[word2idx[START]]], dtype=torch.long, device=device)

    result_ids = []
    model.eval()
    with torch.no_grad():
        for _ in range(max_len):
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src_in, output)
            predictions, *_ = model(src_in, output, enc_mask, dec_enc_mask, dec_mask)
            predicted_id = predictions[0, -1].softmax(dim=-1).argmax(dim=-1).item()
            if predicted_id == word2idx[END]:
                break
            result_ids.append(predicted_id)
            output = torch.cat([output, torch.tensor([[predicted_id]], device=device)], dim=1)

    return " ".join(idx2word[i] for i in result_ids)


sample_questions = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야.",
]

sample_answers = []
for q in sample_questions:
    ans = translate_chatbot(chatbot_transformer, q, word2idx, idx2word)
    sample_answers.append(ans)
    print(">> Q:", q)
    print(">> A:", ans, "<end>\n")


### **Step 7. 성능 측정하기**
---
⚠️ 원본 노트북은 `evaluate()`가 여전히 TF API(`tf.expand_dims`, TF `pad_sequences`)를 쓰고 있었고,
테스트셋에 대한 BLEU 측정도 실제로 호출되지 않았습니다. 아래는 Step 5에서 분리해 둔 테스트셋(`enc_te`, `dec_te`)에 대해
17장에서 정의한 `calculate_bleu()`를 그대로 재사용해 실제로 BLEU Score를 계산합니다.

In [ ]:
def ids_to_tokens(ids, idx2word):
    toks = []
    for i in ids:
        i = int(i)
        if i == word2idx[PAD]:
            continue
        if i == word2idx[END]:
            break
        if i == word2idx[START]:
            continue
        toks.append(idx2word[i])
    return toks


def eval_bleu_chatbot(model, enc_te, dec_te, sample_size=None):
    total_score, n = 0.0, (len(enc_te) if sample_size is None else min(sample_size, len(enc_te)))
    for idx in tqdm(range(n)):
        que_sentence = " ".join(ids_to_tokens(enc_te[idx].tolist(), idx2word))
        reference = ids_to_tokens(dec_te[idx].tolist(), idx2word)
        candidate = translate_chatbot(model, que_sentence, word2idx, idx2word).split()
        if not candidate:
            continue
        total_score += calculate_bleu(reference, candidate)
    print("Num of Sample:", n)
    print("평균 BLEU:", total_score / n)
    return total_score / n


chatbot_bleu = eval_bleu_chatbot(chatbot_transformer, enc_te, dec_te)


### ⚠️ BLEU Score 해석 시 주의
---
BLEU는 원래 번역처럼 정답이 거의 유일한 태스크를 위한 지표입니다. 챗봇은 하나의 질문에 여러 그럴듯한 답이 있을 수 있는
**1:다(one-to-many)** 문제라서, 테스트셋 BLEU는 참고용 보조 지표로만 보고 **Step 6에서 생성한 예문 답변을 사람이 직접 읽고 판단**하는 것이
더 중요합니다.

In [ ]:
print("Translations")
for i, (q, a) in enumerate(zip(sample_questions, sample_answers), 1):
    print(f"> {i}. {a} <end>")

print()
print("Hyperparameters")
print(f"> n_layers: {CHAT_N_LAYERS}")
print(f"> d_model: {CHAT_D_MODEL}")
print(f"> n_heads: {CHAT_N_HEADS}")
print(f"> d_ff: {CHAT_D_FF}")
print(f"> dropout: {CHAT_DROPOUT}")

print()
print("Training Parameters")
print(f"> Warmup Steps: {CHAT_WARMUP_STEPS}")
print(f"> Batch Size: {CHATBOT_BATCH_SIZE}")
print(f"> Epoch At: {CHAT_EPOCHS}")

print()
print(f"BLEU Score(Test set 평균): {chatbot_bleu:.4f}")
